In [1]:
import torch
from torch import nn
from torch import optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

In [2]:
from torch import nn

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [4]:
in_dim = 3
out_dim = 2
hidden_dim = 100
#forse così impara una f.ne non lineare di 2 variabili(vediamo)
class Autoencoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), #0
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim), #1
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, out_dim), #2
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, hidden_dim), #0
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim), #1
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, in_dim), #2 #voglio la RelU dopo questo layer?
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model_1 = Autoencoder(in_dim, hidden_dim, out_dim).to(device)

In [5]:
loaded_model = Autoencoder(in_dim, hidden_dim, out_dim).to(device)
# Load the state_dict of our saved model (this will update the new instance of our model with trained weights)
loaded_model.load_state_dict(torch.load(f='models/Sphere_autoencoder.pth'))

<All keys matched successfully>

In [6]:
#Loading the data back into PyTorch
loaded_data = torch.load('data/sphere_dataset.pt')
loaded_data_enc = torch.load('data/encoder_angles.pt')

X_angles = loaded_data['angles']
Y_cartesian = loaded_data['cartesian']
z_en = loaded_data_enc['encoder_coordinates']


In [7]:
import plotly.graph_objects as go
import numpy as np

# Supponiamo che tu abbia già estratto x, y, z e phi_angles come prima:
x = Y_cartesian[:, 0].cpu().numpy()
y = Y_cartesian[:, 1].cpu().numpy()
z = Y_cartesian[:, 2].cpu().numpy()
phi_angles = X_angles[:, 1].cpu().numpy()

# Creiamo la figura 3D
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers',
    marker=dict(
        size=2,
        color=phi_angles,  # Usiamo phi per il colore
        colorscale='hsv',  # La stessa mappa di colori ciclica
        opacity=0.6,
        colorbar=dict(title="Phi Angle (rad)")
    )
)])

# Miglioriamo il layout per renderlo un cubo perfetto
fig.update_layout(
    title="3D Representation of the Target Sphere (Interactive)",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube' # Mantiene le proporzioni corrette della sfera
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [24]:
#Move to device ONCE to save memory bandwidth
Y_cart_device = Y_cartesian.to(device)

d = Y_cart_device - loaded_model(Y_cart_device)
distances = (d ** 2).sum(dim=1)
threshold = 0.005
#this is like a mask that gives us the indices of the rows where the condition is true
filtered_values = Y_cart_device[distances > threshold]

print(f"Found {len(filtered_values)} values above {threshold}.")
print(filtered_values)

Found 47 values above 0.005.
tensor([[-0.1123,  0.9936, -0.0110],
        [-0.0487,  0.9655,  0.2557],
        [ 0.0561,  0.9707,  0.2338],
        [ 0.1315,  0.9853,  0.1088],
        [ 0.1071,  0.9872,  0.1179],
        [ 0.0220,  0.9773,  0.2108],
        [-0.0901,  0.9800,  0.1772],
        [-0.1505,  0.9885,  0.0181],
        [-0.0977,  0.9940,  0.0483],
        [ 0.1153,  0.9800,  0.1621],
        [ 0.0451,  0.9939,  0.1011],
        [-0.0433,  0.9812,  0.1881],
        [-0.0868,  0.9961,  0.0146],
        [-0.0346,  0.9979, -0.0542],
        [-0.1402,  0.9891,  0.0449],
        [ 0.0727,  0.9817,  0.1762],
        [ 0.0346,  0.9889,  0.1447],
        [-0.1589,  0.9863,  0.0438],
        [-0.0419,  0.9984, -0.0388],
        [-0.0757,  0.9971, -0.0126],
        [-0.0995,  0.9948, -0.0193],
        [-0.1504,  0.9857,  0.0757],
        [ 0.0077,  0.9871,  0.1601],
        [-0.0384,  0.9617,  0.2713],
        [ 0.1107,  0.9876,  0.1112],
        [-0.0313,  0.9703,  0.2397],
        [

In [23]:
# 1. Mettiamo il modello in modalità valutazione ed estraiamo i blocchi
model_1.eval()
encoder = loaded_model.encoder
decoder = loaded_model.decoder

# 2. Calcoliamo i confini effettivi dello spazio latente per sapere dove creare la griglia
with torch.no_grad():
    latents = encoder(Y_cartesian.to(device)).cpu().numpy()

z1_min, z1_max = latents[:, 0].min(), latents[:, 0].max()
z2_min, z2_max = latents[:, 1].min(), latents[:, 1].max()

# 3. Definiamo la densità della griglia (quanti "assi" vogliamo)
num_lines = 15       # Numero di linee per ogni asse coordinato
pts_per_line = 100   # Risoluzione di ogni linea per renderla fluida nel 3D

z1_values = np.linspace(z1_min, z1_max, num_lines)
z2_values = np.linspace(z2_min, z2_max, num_lines)

# 4. Inizializziamo la figura Plotly
fig = go.Figure()

# Nota: Mettiamo i punti originali grigi e trasparenti sullo sfondo per dare un riferimento visivo
x_orig = Y_cartesian[:, 0].cpu().numpy()
y_orig = Y_cartesian[:, 1].cpu().numpy()
z_orig = Y_cartesian[:, 2].cpu().numpy()

fig.add_trace(go.Scatter3d(
    x=x_orig, y=y_orig, z=z_orig,
    mode='markers',
    marker=dict(size=1.5, color='lightgray', opacity=0.15),
    name='Sfera di Riferimento'
))
fig.add_trace(go.Scatter3d(
    x=filtered_values[:, 0].cpu().numpy(), y=filtered_values[:, 1].cpu().numpy(), z=filtered_values[:, 2].cpu().numpy(),
    mode='markers',
    marker=dict(size=2, color='green', opacity=0.6),
    name='Valori distanti dalla ricostruzione'
))

# 5. Generiamo e decodifichiamo le linee degli assi latenti
with torch.no_grad():
    
    # --- ASSE 1: Linee a Z1 costante (varia Z2) -> Colore BLU ---
    for z1 in z1_values:
        line_2d = np.zeros((pts_per_line, 2))
        line_2d[:, 0] = z1
        line_2d[:, 1] = np.linspace(z2_min, z2_max, pts_per_line)
        
        # Passiamo la linea al decoder
        line_tensor = torch.tensor(line_2d, dtype=torch.float32).to(device)
        xyz = decoder(line_tensor).cpu().numpy()
        
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode='lines',
            line=dict(color='rgba(0, 0, 255, 0.7)', width=2.5),
            showlegend=False
        ))

    # --- ASSE 2: Linee a Z2 costante (varia Z1) -> Colore ROSSO ---
    for z2 in z2_values:
        line_2d = np.zeros((pts_per_line, 2))
        line_2d[:, 0] = np.linspace(z1_min, z1_max, pts_per_line)
        line_2d[:, 1] = z2
        
        # Passiamo la linea al decoder
        line_tensor = torch.tensor(line_2d, dtype=torch.float32).to(device)
        xyz = decoder(line_tensor).cpu().numpy()
        
        fig.add_trace(go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode='lines',
            line=dict(color='rgba(255, 0, 0, 0.7)', width=2.5), 
            showlegend=False
        ))

# 6. Configurazione finale del Layout per visualizzare una sfera perfetta
fig.update_layout(
    title="Assi di Coordinata dell'Autoencoder deformati sulla Sfera 3D",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube' # Fondamentale per non distorcere la sfera in un uovo
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()
#cosa succede nel buco, perché il buco è lì?) 
#i punti del dataset sulla sfera come vengono?

# sembrano coordinate stereografiche!!

In [9]:
with torch.no_grad():
    reconstructed = loaded_model(Y_cartesian.to(device))

In [10]:
fig = go.Figure()
x = reconstructed[:, 0].cpu().numpy()
y = reconstructed[:, 1].cpu().numpy()
z = reconstructed[:, 2].cpu().numpy()
fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=1.5, color='red', opacity=0.15),
    name='Sfera di Riferimento'
))

# proviamo a vedere se sono coordinate stereografiche

In [ ]:
def coord_stereografiche(x):
    # x è un tensore di forma (N, 3) con coordinate cartesiane (x, y, z)
    x_cart = x[:, 0]
    y_cart = x[:, 1]
    z_cart = x[:, 2]
  
    denominator = 1 - z_cart + 1.e-10
    u = x_cart / denominator
    v = y_cart / denominator
    
    return torch.stack((u, v), dim=1)

In [15]:
class StereographicProjection:
    def __init__(self):
        super().__init__() 
        self.rotation = nn.Linear(3, 3, bias=False)  #rotazione 3D, bias=False perché no traslazioni!!
        self.lin_corr = nn.Linear(2, 2)  #regressione lineare per vedere se è stereografica!!!
        torch.transpose(self.rotation.weight) == torch.inverse(self.rotation.weight + 1.e-10*torch.eye(3)) #inizializzo la matrice di rotazione come matrice di identità (o quasi, per evitare problemi di inversione)

    def forward(self, x):
        rotated = self.rotation(x)
        projected = coord_stereografiche(rotated)
        z_hopefully = self.lin_corr(projected)
        return z_hopefully

In [ ]:
epochs = 10

train_loss_values = []
test_loss_values = []
epoch_count = []

for epoch in range(epochs):
    sum_loss = 0.0
    for x, y in train_loader_1:
        x = x.to(device)
        y = y.to(device)
        model_0.train()
        y_pred = model_0(x)
        loss = loss_fn(y_pred, y)
        sum_loss += loss.item()
        optimizer_0.zero_grad()
        loss.backward()
        optimizer_0.step()

    avg_loss = sum_loss / len(train_loader_1)
    with torch.no_grad(): 
      model_0.eval()
      for x, y in test_loader_1:
        test_features = x.to(device)
        test_labels = y.to(device)
      test_pred = model_0(test_features)
      test_loss = loss_fn(test_pred, test_labels)
      epoch_count.append(epoch)
      train_loss_values.append(avg_loss)
      test_loss_values.append(test_loss.detach().cpu().numpy())
      print(f"Epoch: {epoch} | MSE Train Loss: {avg_loss} | MSE Test Loss: {test_loss} ")